In [1]:
import numpy as np
from sklearn.metrics import f1_score, classification_report
from typing import Tuple, Iterable, Dict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import Module
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import MNIST
import math

In [2]:
BATCH_SIZE: int = 64

train = MNIST(root=".", download=True, train=True)
test = MNIST(root=".", download=True, train=False)

In [3]:
class MNISTTrainingSet(Dataset):
    def __init__(self, device: torch.device) -> None:
        self._device = device

        X_train, y_train = (
            [],
            [],
        )

        for image, label in train:
            image_tensor: torch.Tensor = torch.Tensor(np.asarray(image))
            label_tensor: torch.Tensor = torch.Tensor([label]).long()

            X_train.append(image_tensor)
            y_train.append(label_tensor)

        self._X = torch.stack(X_train).to(self._device)
        self._y = torch.stack(y_train).to(self._device)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self._X[index], self._y[index]

    def __len__(self) -> int:
        return len(self._X)

    def get_X(self) -> torch.Tensor:
        return self._X


class MNISTTestingSet(Dataset):
    def __init__(self, device: torch.device) -> None:
        self._device = device

        X_test, y_test = [], []

        for image, label in test:
            image_tensor: torch.Tensor = torch.Tensor(np.asarray(image))
            label_tensor: torch.Tensor = torch.Tensor([label]).long()

            X_test.append(image_tensor)
            y_test.append(label_tensor)

            self._X = torch.stack(X_test).to(self._device)
            self._y = torch.stack(y_test).to(self._device)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self._X[index], self._y[index]

    def __len__(self) -> int:
        return len(self._X)

    def get_X(self) -> torch.Tensor:
        return self._X


if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)
training_set = MNISTTrainingSet(device)
testing_set = MNISTTestingSet(device)

print(len(training_set), len(testing_set))

mps


/var/folders/q6/0q0r0xgd16d478qhsd_5hhlc0000gn/T/ipykernel_93176/1082707506.py:11: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  image_tensor: torch.Tensor = torch.Tensor(np.asarray(image))


60000 10000


In [4]:
"""
    For reproducicibility a seed is manually chosen.
"""

generator: torch.Generator = torch.Generator()
generator.manual_seed(17)

train_loader: DataLoader = DataLoader(
    training_set, batch_size=BATCH_SIZE, generator=generator
)
test_loader: DataLoader = DataLoader(
    testing_set, batch_size=BATCH_SIZE, generator=generator
)

In [5]:
def train_then_eval(model, epochs: int = 5, lr: float = 1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()

            predictions = model(batch_X)

            loss = criterion(predictions, batch_y.squeeze())
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

    print(f"Entraînement terminé. {len(train_losses)} itérations enregistrées.")

    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            predictions = model(batch_X)

            preds_classes = predictions.argmax(dim=1)

            all_preds.extend(preds_classes.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())

    f1_global = f1_score(all_targets, all_preds, average="macro")
    accuracy = np.mean(np.array(all_preds) == np.array(all_targets))

    print(classification_report(all_targets, all_preds, digits=4))
    print(f"F1-score global (macro) : {f1_global:.4f}")
    print(f"Précision globale (Accuracy) : {accuracy:.4f}")

In [6]:
class MLP(Module):
    def __init__(
        self,
        network_size: Iterable[int],
        device: torch.device,
        dropout: float = 0,
    ) -> None:
        super().__init__()

        self._device = device
        self.dropout = dropout
        sub_models = [
            nn.Flatten(),
        ]
        if dropout == 0:
            for k in range(0, len(network_size) - 1):
                sub_models.append(nn.ReLU())
                sub_models.append(nn.BatchNorm1d(network_size[k], device=self._device))
                sub_models.append(
                    nn.Linear(network_size[k], network_size[k + 1], device=self._device)
                )
        else:
            for k in range(0, len(network_size) - 1):
                sub_models.append(nn.ReLU())
                sub_models.append(nn.Dropout(self.dropout))
                sub_models.append(nn.BatchNorm1d(network_size[k], device=self._device))
                sub_models.append(
                    nn.Linear(network_size[k], network_size[k + 1], device=self._device)
                )

        self._model = nn.Sequential(*sub_models)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return self._model(input)

In [42]:
model = MLP([784, 120, 80, 10], device)

train_then_eval(model)

Entraînement terminé. 4690 itérations enregistrées.
              precision    recall  f1-score   support

           0     0.9756    0.9776    0.9766       980
           1     0.9567    0.9930    0.9745      1135
           2     0.9605    0.9419    0.9511      1032
           3     0.9543    0.9505    0.9524      1010
           4     0.9742    0.9623    0.9682       982
           5     0.9224    0.9731    0.9471       892
           6     0.9677    0.9395    0.9534       958
           7     0.9489    0.9582    0.9535      1028
           8     0.9795    0.9322    0.9553       974
           9     0.9429    0.9495    0.9462      1009

    accuracy                         0.9581     10000
   macro avg     0.9583    0.9578    0.9578     10000
weighted avg     0.9584    0.9581    0.9581     10000

F1-score global (macro) : 0.9578
Précision globale (Accuracy) : 0.1004


In [8]:
class CNN(Module):
    def __init__(
        self,
        convolution_params: Dict[str, Iterable[int]],
        mlp_network_size: Iterable[int],
        device: torch.device,
    ) -> None:
        super().__init__()

        self._device = device
        self._convolution_params = convolution_params
        sub_models = []
        num_conv_layers = len(convolution_params["channels"]) - 1

        for k in range(num_conv_layers):
            sub_models.append(
                nn.Conv2d(
                    in_channels=convolution_params["channels"][k],
                    out_channels=convolution_params["channels"][k + 1],
                    kernel_size=convolution_params["kernel_size"][k],
                    stride=convolution_params["stride"][k],
                    padding=convolution_params["padding"][k],
                    dilation=convolution_params["dilation"][k],
                    device=self._device,
                )
            )
            sub_models.append(nn.ReLU())
            sub_models.append(nn.MaxPool2d(convolution_params["pooling_size"][k]))

        sub_models.append(nn.Flatten())
        sub_models.append(MLP(mlp_network_size, self._device))

        self._model = nn.Sequential(*sub_models)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        input = input.view(-1, 1, 28, 28)
        return self._model(input)

In [9]:
cnn_params = {
    "channels": [1, 4, 16],
    "kernel_size": [3, 3],
    "stride": [1, 1],
    "padding": [0, 0],
    "dilation": [1, 1],
    "pooling_size": [2, 2],
}

model = CNN(cnn_params, [400, 32, 10], device)

train_then_eval(model, epochs=10)

Entraînement terminé. 9380 itérations enregistrées.
              precision    recall  f1-score   support

           0     0.9788    0.9888    0.9838       980
           1     0.9826    0.9965    0.9895      1135
           2     0.9872    0.9719    0.9795      1032
           3     0.9871    0.9832    0.9851      1010
           4     0.9858    0.9868    0.9863       982
           5     0.9841    0.9742    0.9792       892
           6     0.9821    0.9749    0.9785       958
           7     0.9750    0.9844    0.9797      1028
           8     0.9844    0.9713    0.9778       974
           9     0.9736    0.9851    0.9793      1009

    accuracy                         0.9820     10000
   macro avg     0.9821    0.9817    0.9819     10000
weighted avg     0.9820    0.9820    0.9820     10000

F1-score global (macro) : 0.9819
Précision globale (Accuracy) : 0.1004


In [39]:
class MultiHeadAttention(Module):
    def __init__(self, d_model, number_heads, device):
        super().__init__()

        assert d_model % number_heads == 0

        self.d_k = d_model // number_heads
        self.number_heads = number_heads
        self.d_model = d_model

        self.Q = nn.Linear(d_model, d_model, device=device)
        self.K = nn.Linear(d_model, d_model, device=device)
        self.V = nn.Linear(d_model, d_model, device=device)

    def forward(self, queries, keys, values):
        batch, query_length, _ = queries.shape

        queries = (
            self.Q(queries)
            .view(batch, query_length, self.number_heads, self.d_k)
            .transpose(1, 2)
        )
        keys = (
            self.K(keys)
            .view(batch, query_length, self.number_heads, self.d_k)
            .transpose(1, 2)
        )
        values = (
            self.V(values)
            .view(batch, query_length, self.number_heads, self.d_k)
            .transpose(1, 2)
        )

        scores = torch.softmax(
            queries @ keys.transpose(-1, -2) / math.sqrt(self.d_k), dim=-1
        )

        return (
            (scores @ values)
            .transpose(1, 2)
            .contiguous()
            .view(batch, query_length, self.number_heads * self.d_k)
        )


class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim, device):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(d_model, hidden_dim, device=device),
            nn.ReLU(),
            nn.Linear(hidden_dim, d_model, device=device),
        )

    def forward(self, x):
        return self.model(x)


class Encoder(Module):
    def __init__(self, d_model, number_heads, hidden_dim, device):
        super().__init__()

        self.self_attention = MultiHeadAttention(d_model, number_heads, device=device)
        self.norm1 = nn.LayerNorm(d_model, device=device)
        self.feed_forward = FeedForward(d_model, hidden_dim, device=device)
        self.norm2 = nn.LayerNorm(d_model, device=device)

    def forward(self, input):
        x = self.norm1(self.self_attention(input, input, input) + input)
        return self.norm2(self.feed_forward(x) + x)


class ImageDecomposer(Module):
    def __init__(self, d_model: int, kernel_size: int, device):
        super().__init__()

        self.d_model = d_model
        self.kernel_size = kernel_size

        self.decomposer = nn.Conv2d(
            in_channels=1,
            out_channels=self.d_model,
            kernel_size=self.kernel_size,
            stride=kernel_size,
            device=device,
        )

    def forward(self, input) -> torch.Tensor:
        x = self.decomposer(input)
        x = x.flatten(start_dim=2)
        return x.transpose(-2, -1)


class MNISTTransformer(Module):
    def __init__(
        self,
        d_model,
        autre,
        kernel_size,
        number_heads,
        hidden_dim,
        number_encoders: int,
        device: torch.device = torch.device("cpu"),
    ):
        super().__init__()

        self.device = device

        self.d_model = d_model
        self.number_heads = number_heads
        self.hidden_dim = hidden_dim

        self.decomposer = ImageDecomposer(d_model, kernel_size, device=self.device)

        self.positionnal_encoder = nn.Parameter(
            torch.randn(1, autre, self.d_model, device=self.device)
        )

        self.encoder = nn.Sequential(
            *[
                Encoder(
                    self.d_model, self.number_heads, self.hidden_dim, device=self.device
                )
                for _ in range(number_encoders)
            ]
        )

        self.classifier = nn.Linear(self.d_model, 10, device=self.device)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        x = input.view(-1, 1, 28, 28)
        x = self.encoder(self.decomposer(x) + self.positionnal_encoder)
        x = x.mean(dim=1)
        return self.classifier(x)


model = MNISTTransformer(64, 16, 7, 32, 128, 4, device)

In [ ]:
train_then_eval(model, epochs=20, lr=5e-4)

Entraînement terminé. 32830 itérations enregistrées.
              precision    recall  f1-score   support

           0     0.9866    0.9755    0.9810       980
           1     0.9323    0.9947    0.9625      1135
           2     0.9676    0.9554    0.9615      1032
           3     0.9602    0.9564    0.9583      1010
           4     0.9803    0.9633    0.9718       982
           5     0.9529    0.9742    0.9634       892
           6     0.9783    0.9405    0.9590       958
           7     0.9503    0.9679    0.9590      1028
           8     0.9800    0.9548    0.9672       974
           9     0.9610    0.9534    0.9572      1009

    accuracy                         0.9640     10000
   macro avg     0.9650    0.9636    0.9641     10000
weighted avg     0.9644    0.9640    0.9640     10000

F1-score global (macro) : 0.9641
Précision globale (Accuracy) : 0.1005
